In [1]:
import os

In [2]:
%pwd

'd:\\MLOPS\\Projects\\DataScienceProject1\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\MLOPS\\Projects\\DataScienceProject1'

In [9]:
from dataclasses import dataclass
from pathlib import Path

@dataclass # used for assign the values to the variables in the class
class DataIngestionConfig:
  root_dir: Path
  source_URL: str
  local_data_file: Path
  unzip_dir: Path

In [5]:
from src.DataScienceProject.constants import *

In [17]:
from src.DataScienceProject.utils.common import read_yaml , create_directories
import zipfile


In [22]:
class ConfigurationManager:
  def __init__(self , config_filepath=CONFIG_FILE_PATH , params_filepath=PARAMS_FILE_PATH , schema_filepath=SCHEMA_FILE_PATH):
    self.config = read_yaml(config_filepath)
    self.params = read_yaml(params_filepath)
    self.schema = read_yaml(schema_filepath)
    
    create_directories([self.config.artifacts_root])
  # downloading the zip file
  def data_ingestion_config(self) -> DataIngestionConfig:
    config = self.config.data_ingetion
    create_directories([config.root_dir])
    data_ingestion_config = DataIngestionConfig(
      root_dir = config.root_dir,
      source_URL = config.source_URL,
      local_data_file = config.local_data_file,
      unzip_dir = config.unzip_dir
    )
    
    return data_ingestion_config
  
  def extract_zip_file(self , zip_file_path:Path , unzip_dir:Path):
    with zipfile.ZipFile(zip_file_path , 'r') as zip_ref:
      zip_ref.extractall(unzip_dir) 
    

In [14]:
import urllib.request as request
from src.DataScienceProject import logger

In [ ]:
# Component
class DataIngetion:
  def __init__(self , config: DataIngestionConfig):
    self.config = config
  
  def download_self(self):
    if not os.path.exists(self.config.local_data_file):
      filename , headers = request.urlretrieve(url=self.config.source_URL , filename=self.config.local_data_file)
      logger.info(f"File is downloaded at location {filename} with info {headers}")
    else:
      print(f"File already exists of path {self.config.local_data_file}")
  

In [23]:
try:
  config=ConfigurationManager()
  data_ingestion_config = config.data_ingestion_config()
  data_ingestion = DataIngetion(config=data_ingestion_config)
  data_ingestion.download_self()
  config.extract_zip_file(zip_file_path=data_ingestion_config.local_data_file , unzip_dir=data_ingestion_config.unzip_dir)
except Exception as e:
  logger.exception(e)

[2026-05-20 01:55:47,688]: INFO: common :yaml file: config\config.yaml loaded successfully 
[2026-05-20 01:55:47,689]: INFO: common :yaml file: params.yaml loaded successfully 
[2026-05-20 01:55:47,691]: INFO: common :yaml file: schema.yaml loaded successfully 
[2026-05-20 01:55:47,692]: INFO: common :created directory at: artifacts 
[2026-05-20 01:55:47,693]: INFO: common :created directory at: artifacts/data_ingestion 
File already exists of path artifacts/data_ingestion/data.zip
